# TMC-PINN Experiments — FP64 Adam → FP64 L-BFGS ablation

Identical to `run_experiments.ipynb` except that each PDE cell runs **one**
condition instead of all seven: `fp64adam_to_fp64lbfgs`, the only cell still
marked `[PLACEHOLDER]` in the frozen-facts table.

Nothing in `train_pinn.py` changes. `run_all` already takes a `conditions`
argument; this notebook just passes one entry instead of the default seven.

| PDE | Epochs | Switch @ | Cell |
|---|---|---|---|
| Reaction | 2,000 | 1,000 | 5 |
| Wave | 10,000 | 5,000 | 6 |
| Allen-Cahn (1:1:1) | 10,000 | 5,000 | 7 |
| Allen-Cahn (Xu et al. weighted) | 10,000 | 5,000 | 7b |
| Burgers' | 10,000 | 5,000 | 8 |

**Cost.** Each run is Adam for the first half of the budget and L-BFGS for the
second, so expect roughly the wall-clock of that PDE's existing
`fp32adam_to_fp64lbfgs` run — e.g. AC-weighted took 11,506 s there. The FP64
Adam half is somewhat slower than the FP32 Adam half, so treat that as a floor.

**Results land in `./results/<pde>/fp64adam_to_fp64lbfgs/`** — a label that does
not yet exist, so no existing run can be overwritten.

**Compute budget:** ~$3.29/hr on H100 PCIe. Confirm Lambda balance before starting long runs.

In [ ]:
# Cell 1 — GPU check
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM    : {props.total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be very slow')

In [ ]:
# Cell 2 — Clone / update repo
import os

if os.path.exists('PInnns'):
    os.chdir('PInnns')
    os.system('git pull origin main')
else:
    os.system('git clone https://github.com/michae6345-crypto/PInnns.git')
    os.chdir('PInnns')

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Cell 3 — Load train_pinn.py
import importlib.util, sys

if 'train_pinn' in sys.modules:
    del sys.modules['train_pinn']

spec = importlib.util.spec_from_file_location('train_pinn', './train_pinn.py')
tp   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tp)

print('train_pinn.py loaded')
print('Available PDEs:', list(tp.PDE_BUILDERS.keys()))
print('Epoch budgets: ', tp.PDE_EPOCHS)

In [ ]:
# Cell 4 — Smoke test (~1 min, 50 epochs)
# Run this first every session to confirm the environment is working.
tp.main(
    pde          = 'reaction',
    dtype_start  = 'fp64',
    optim_start  = 'adam',
    total_epochs = 50,
    out_dir      = './results_smoke'
)
print('Smoke test passed.')

In [ ]:
# Cell 4b — the one condition this notebook runs
# Same dict shape as the entries in tp.ALL_CONDITIONS.
# dtype is fp64 throughout; only the optimizer switches, at PDE_SWITCH[pde].
# run_all derives the label 'fp64adam_to_fp64lbfgs' from these four fields.
ABLATION = [
    dict(dtype_start='fp64', optim_start='adam',
         dtype_switch='fp64', optim_switch='lbfgs'),
]

# Sanity check: label and switch epochs, before anything trains.
c = ABLATION[0]
label = f"{c['dtype_start']}{c['optim_start']}_to_{c['dtype_switch']}{c['optim_switch']}"
print('Condition label:', label)
assert label == 'fp64adam_to_fp64lbfgs'
for pde in ['reaction', 'wave', 'ac', 'ac_weighted', 'burgers']:
    print(f'  {pde:<12} {tp.PDE_EPOCHS[pde]:>6} epochs   switch @ {tp.PDE_SWITCH[pde]}')

---
## Main PDE Runs
Each cell runs the single `fp64adam_to_fp64lbfgs` condition for one PDE.
`skip_existing=True` means an already-completed run is skipped safely if the
kernel dies mid-run.

In [ ]:
# Cell 5 — REACTION  (2,000 epochs x 1 condition, switch @ 1,000)
# u_t = rho*u*(1-u), rho=5.  Epoch budget from Xu et al.
tp.run_all(
    pdes          = ['reaction'],
    conditions    = ABLATION,
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 6 — WAVE  (10,000 epochs x 1 condition, switch @ 5,000)
# u_tt = 4*u_xx.  Epoch budget from Xu et al.
tp.run_all(
    pdes          = ['wave'],
    conditions    = ABLATION,
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 7 — ALLEN-CAHN  (10,000 epochs x 1 condition, switch @ 5,000)
# u_t - 0.0001*u_xx + 5(u^3-u) = 0.  IC: x^2*cos(pi*x).  Loss: 1:1:1 (unweighted).
# allen_cahn.mat must be in repo root.
tp.run_all(
    pdes          = ['ac'],
    conditions    = ABLATION,
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 7b — ALLEN-CAHN WEIGHTED  (10,000 epochs x 1 condition, switch @ 5,000)
# Same PDE as Cell 7 but with Xu et al. exact loss weighting: 10*res + bc + 100*ic.
# Results saved to results/ac_weighted/ — never overwrites ac (1:1:1) results.
tp.run_all(
    pdes          = ['ac_weighted'],
    conditions    = ABLATION,
    out_dir       = './results',
    skip_existing = True
)

In [ ]:
# Cell 8 — BURGERS'  (10,000 epochs x 1 condition, switch @ 5,000)
# u_t + u*u_x - (0.01/pi)*u_xx = 0,  x in [-1,1],  t in [0,1]
# IC: u(x,0) = -sin(pi*x)   BC: u(-1,t) = u(1,t) = 0  (Dirichlet)
# Epoch budget is our own choice -- no Xu et al. precedent. Disclose in paper.
# burgers_shock.mat must be in repo root.
tp.run_all(
    pdes          = ['burgers'],
    conditions    = ABLATION,
    out_dir       = './results',
    skip_existing = True
)

---
## Utilities

In [ ]:
# Cell 9 — Check results summary (this ablation only)
import os, pandas as pd

CONDITIONS = ['fp64adam_to_fp64lbfgs']
PDES = ['reaction', 'wave', 'ac', 'ac_weighted', 'burgers']

print(f'{"PDE":<14} {"Condition":<35} {"L2":>12} {"Time(s)":>10}')
print('-'*75)
for pde in PDES:
    for cond in CONDITIONS:
        path = f'./results/{pde}/{cond}/{pde}_PINN_{cond}_eval.csv'
        if os.path.exists(path):
            try:
                row = pd.read_csv(path).iloc[-1]
                print(f'{pde:<14} {cond:<35} {row["L2_rel"]:>12.4e} {row["total_time_s"]:>10.0f}')
            except Exception as e:
                print(f'{pde:<14} {cond:<35} ERROR: {e}')
        else:
            print(f'{pde:<14} {cond:<35} NOT RUN')

In [ ]:
# Cell 10 — Generate paper figures


In [ ]:
# Cell 11 — Backup to zip (download before closing Lambda)
